# 🧬 Kodomo Exercises: FASTA & Sequence Analysis
# Упражнения Кодомо: FASTA и анализ последовательностей

---

Based on MSU Kodomo Bioinformatics Program (pr8)

## 🎯 Learning Objectives

- Parse FASTA files correctly
- Calculate GC content
- Extract sequence names
- Count CDS features in GenBank files
- Classify nucleotides (purine/pyrimidine)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This assignment notebook is designed for hands-on practice of production-relevant analysis patterns.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Start with tiny inputs first, then scale after outputs look correct.
- Document assumptions for every threshold or filtering decision.
- Debug shape/type/path issues early to avoid cascading errors.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

## 📊 FASTA Format Review

```
┌─────────────────────────────────────────────────────────────────┐
│                      FASTA FILE STRUCTURE                       │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  >sp|P12345|GENE_HUMAN Description of the protein               │
│  ↑  ↑      ↑          ↑                                         │
│  │  │      │          └── Free text description                 │
│  │  │      └───────────── Gene/protein name                     │
│  │  └──────────────────── Accession number                      │
│  └─────────────────────── Database prefix (sp=SwissProt)        │
│                                                                 │
│  MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQE │
│  EYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHQYREQIKRVKDSDDVPMVLVGNKCDLAAR  │
│  ↑                                                              │
│  └── Sequence lines (no spaces, uppercase)                      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Exercise 1: Extract FASTA Names

**Task:** Write a function that extracts all sequence names from a FASTA file.

Original Kodomo solution (Python 2):
```python
# Kravchenko_pr8_fastanames.py
from sys import argv
indata = open(argv[1], "r")
line = indata.readline()
while line != "":
    if line[0] == ">":
        print line[1:].strip()
    line = indata.readline()    
indata.close()
```

In [ ]:
# Modern Python 3 Solution

def extract_fasta_names(filename):
    """
    Extract sequence names from FASTA file.
    
    Parameters:
        filename: Path to FASTA file
        
    Returns:
        List of sequence names (without '>')
    """
    names = []
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('>'):
                names.append(line[1:].strip())
    return names

# Test with sample data
sample_fasta = '''>
sp|P12345|BRCA1_HUMAN Breast cancer type 1 susceptibility protein
MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQE
>sp|Q9Y6K9|TP53_HUMAN Cellular tumor antigen p53
MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP
'''

# Write test file
with open('/tmp/test.fasta', 'w') as f:
    f.write(sample_fasta)

# Test function
names = extract_fasta_names('/tmp/test.fasta')
for name in names:
    print(name)

---

## Exercise 2: Calculate GC Content

**Task:** Calculate the GC content percentage of a DNA sequence in a FASTA file.

```
┌─────────────────────────────────────────────────────────────────┐
│                     GC CONTENT FORMULA                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│          (count of G) + (count of C)                            │
│  GC% = ─────────────────────────────── × 100                    │
│              total sequence length                              │
│                                                                 │
│  Example: ATGCGATCGA                                            │
│           G=2, C=2, Total=10                                    │
│           GC% = (2+2)/10 × 100 = 40%                            │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

Original Kodomo solution:

In [ ]:
# Original Kodomo approach (Kravchenko_pr8_gc.py) - modernized

def gc_content_kodomo(filename):
    """
    Calculate GC content from FASTA file.
    Based on Kodomo pr8 exercise.
    """
    sequence = ""
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line.startswith('>'):
                sequence += line.upper()
    
    if len(sequence) == 0:
        return 0.0
    
    g_count = sequence.count('G')
    c_count = sequence.count('C')
    
    gc_percent = (g_count + c_count) / len(sequence) * 100
    return gc_percent

# Test
test_fasta = """>test_sequence
ATGCGATCGATCGATCG
ATGCATGCATGCATGCA
"""

with open('/tmp/test_gc.fasta', 'w') as f:
    f.write(test_fasta)

gc = gc_content_kodomo('/tmp/test_gc.fasta')
print(f"GC Content: {gc:.2f}%")

---

## Exercise 3: Nucleotide Classification

**Task:** Classify a nucleotide as purine (A, G) or pyrimidine (C, T, U).

```
┌─────────────────────────────────────────────────────────────────┐
│                    NUCLEOTIDE CLASSIFICATION                     │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  PURINES (двухкольцевые)         PYRIMIDINES (однокольцевые)   │
│  ┌───────────────────┐           ┌───────────────────┐         │
│  │  Adenine (A)      │           │  Cytosine (C)     │         │
│  │  Guanine (G)      │           │  Thymine (T)      │         │
│  │                   │           │  Uracil (U)       │         │
│  └───────────────────┘           └───────────────────┘         │
│                                                                 │
│  Base pairing: A═T (2 H-bonds), G≡C (3 H-bonds)                │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Based on Kravchenko_pr8_nucleobase.py

def classify_nucleotide(nucleotide):
    """
    Classify nucleotide as purine or pyrimidine.
    
    Parameters:
        nucleotide: Single character (A, T, G, C, U)
        
    Returns:
        'purine', 'pyrimidine', or 'invalid'
    """
    nuc = nucleotide.upper()
    
    purines = {'A', 'G'}
    pyrimidines = {'T', 'C', 'U'}
    
    if nuc in purines:
        return 'purine'
    elif nuc in pyrimidines:
        return 'pyrimidine'
    else:
        return 'invalid'

# Test all nucleotides
for nuc in ['A', 'T', 'G', 'C', 'U', 'X']:
    print(f"{nuc}: {classify_nucleotide(nuc)}")

---

## Exercise 4: Count CDS Features

**Task:** Count the number of CDS (Coding Sequence) features in a GenBank file.

Based on `Kravchenko_pr8_cds.py`

In [ ]:
def count_cds(filename):
    """
    Count CDS features in a GenBank file.
    
    Parameters:
        filename: Path to GenBank file
        
    Returns:
        Number of CDS features
    """
    cds_count = 0
    
    with open(filename, 'r') as f:
        for line in f:
            # CDS typically appears at start of feature line
            if '     CDS' in line or line.strip().startswith('CDS'):
                cds_count += 1
    
    return cds_count

# Example GenBank format
print("""
GenBank CDS feature example:

     CDS             join(1..100,200..500)
                     /gene="BRCA1"
                     /protein_id="NP_007294.2"
""")

---

## Exercise 5: Statistical Functions

**Task:** Calculate median and average from a file of numbers.

Based on `Kravchenko_pr8_median.py` and `Kravchenko_pr8_average.py`

In [ ]:
def calculate_stats(numbers):
    """
    Calculate basic statistics.
    
    Parameters:
        numbers: List of numbers
        
    Returns:
        Dictionary with mean, median, min, max
    """
    if not numbers:
        return None
    
    sorted_nums = sorted(numbers)
    n = len(sorted_nums)
    
    # Mean
    mean = sum(numbers) / n
    
    # Median
    if n % 2 == 1:
        median = sorted_nums[n // 2]
    else:
        median = (sorted_nums[n // 2 - 1] + sorted_nums[n // 2]) / 2
    
    return {
        'mean': mean,
        'median': median,
        'min': min(numbers),
        'max': max(numbers),
        'count': n
    }

# Test
test_numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
stats = calculate_stats(test_numbers)
print(f"Statistics for {test_numbers}:")
for key, value in stats.items():
    print(f"  {key}: {value}")

---

## 🏋️ Practice Exercises

### Exercise A: Multi-FASTA Parser
Write a function that parses a multi-FASTA file and returns a dictionary `{name: sequence}`.

### Exercise B: GC Window
Calculate GC content in sliding windows across a sequence.

### Exercise C: Purine/Pyrimidine Ratio
Calculate the ratio of purines to pyrimidines in a sequence.

---

## 📚 Key Takeaways

1. **FASTA parsing**: Always check for `>` at line start
2. **GC content**: Simple counting divided by length
3. **File handling**: Use `with open()` for automatic cleanup
4. **String methods**: `.upper()`, `.strip()`, `.count()` are essential